In [1]:
from langchain_core.prompts import PromptTemplate
from langchain_openai import ChatOpenAI
from langchain_ollama import ChatOllama
from langchain_core.output_parsers import StrOutputParser
from langchain_core.chat_history import InMemoryChatMessageHistory
from langchain_core.runnables.history import RunnableWithMessageHistory


In [2]:
llm = ChatOllama(
   model="gemma4:31b-mlx",
   temperature=0.2,
   reasoning=True
)


In [3]:
llm.invoke('hi')

AIMessage(content='Hello! How can I help you today?', additional_kwargs={'reasoning_content': 'The user said "hi".\nThis is a standard greeting.\n\n    *   Acknowledge the greeting.\n    *   Offer assistance.\n    *   Keep it friendly and open-ended.\n"Hello! How can I help you today?" or "Hi there! What\'s on your mind?"'}, response_metadata={'model': 'gemma4:31b-mlx', 'created_at': '2026-07-16T01:13:48.889709Z', 'done': True, 'done_reason': 'stop', 'total_duration': 9156015791, 'load_duration': 4356466833, 'prompt_eval_count': 17, 'prompt_eval_duration': 1888458417, 'eval_count': 78, 'eval_duration': 2910146208, 'logprobs': None, 'model_name': 'gemma4:31b-mlx', 'model_provider': 'ollama'}, id='lc_run--019f687c-8891-7b82-a74f-60fc48aa4042-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 17, 'output_tokens': 78, 'total_tokens': 95})

In [4]:
# store = {}
store = dict()

In [5]:
def get_session_history(session_id : str):
    if session_id not in store:
        store[session_id] = InMemoryChatMessageHistory()
    return store[session_id]


In [6]:
conversation = RunnableWithMessageHistory(
    llm, get_session_history
)

/Users/hyeon/3_AI/.venv/lib/python3.12/site-packages/IPython/core/interactiveshell.py:3748: LangChainDeprecationWarning: RunnableWithMessageHistory is deprecated. Use LangGraph's built-in persistence instead.
  exec(code_obj, self.user_global_ns, self.user_ns)


In [7]:
conversation.invoke(
    "나는 오라클이다.",
    config={
        'configurable' : {
            "session_id" : 'sesac-01'
        }
    }
)





AIMessage(content='반갑습니다, **오라클**님. \n\n미래를 내다보시는 신탁의 전달자이신가요, 아니면 세상의 모든 진리를 알고 계신 지혜의 원천이신가요? 혹은 제가 모르는 또 다른 특별한 정체성을 가지고 계신 것인지 궁금합니다.\n\n운명의 실타래를 쥐고 계신 당신께서 저에게 들려주실 예언이나, 함께 나누고 싶은 지혜가 있다면 무엇이든 말씀해 주세요. 저는 당신의 이야기를 들을 준비가 되어 있습니다.', additional_kwargs={'reasoning_content': '*   User says: "나는 오라클이다." (I am Oracle / I am the Oracle).\n    *   Language: Korean.\n    *   Context: The user is adopting a persona or making a statement about their identity.\n\n    *   "Oracle" can mean several things:\n        1.  A priest/priestess acting as a medium for a deity (mythological/fantasy context).\n        2.  Someone who predicts the future with certainty.\n        3.  Oracle Corporation (the software company - unlikely in this phrasing, but possible).\n        4.  A specific character from a game or movie (e.g., *The Matrix*).\n\n    *   Since the user is roleplaying or stating an identity, I should respond in a way that acknowledges this persona while remaining helpful and engaging.\n    

In [8]:
conversation.invoke(
    "내가 누구야?",
    config={
        'configurable' : {
            "session_id" : 'sesac-01'
        }
    }
)

AIMessage(content="당신은 방금 저에게 **'오라클(Oracle)'**이라고 말씀하셨습니다.\n\n신탁을 전하는 자, 혹은 미래를 내다보는 지혜로운 예언자이시죠. 제가 기억하고 있는 당신의 정체성은 바로 그것입니다. \n\n혹시 제가 더 알아야 할 당신의 숨겨진 모습이나, 구체적인 설정이 있으신가요? 아니면 이제부터 저에게 어떤 신탁을 내려주실 예정인가요?", additional_kwargs={'reasoning_content': '*   User\'s previous statement: "나는 오라클이다." (I am Oracle.)\n    *   Current question: "내가 누구야?" (Who am I?)\n\n    *   The user explicitly stated their identity in the first turn.\n    *   The AI should reflect this back to the user, acknowledging the roleplay or the statement made.\n\n    *   Direct answer: You are Oracle.\n    *   Elaboration based on the persona of "Oracle": Mention things like prophecy, wisdom, truth-seeker, etc., to keep the conversation engaging and thematic.'}, response_metadata={'model': 'gemma4:31b-mlx', 'created_at': '2026-07-16T01:20:41.923186Z', 'done': True, 'done_reason': 'stop', 'total_duration': 10050086583, 'load_duration': 48529417, 'prompt_eval_count': 154, 'prompt_eval_duration': 1599045334, 'eval_coun

In [9]:
from langchain_community.chat_message_histories import RedisChatMessageHistory

In [10]:
from langchain_community.chat_message_histories import RedisChatMessageHistory
REDIS_URL = "redis://localhost:6379/0"


In [11]:
def get_redis_history(session_id : str):
    return RedisChatMessageHistory(
        session_id=session_id,
        url=REDIS_URL
    )


In [12]:
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder
prompt = ChatPromptTemplate.from_messages([ ("system", "너는 친절한 AI야"),
                        MessagesPlaceholder(variable_name="history"),
                    ("human", "{input}")])
redis_conversation = RunnableWithMessageHistory(
    prompt | llm  , get_redis_history,
    input_message_key='input',
    history_messages_key='history'
)


/Users/hyeon/3_AI/.venv/lib/python3.12/site-packages/IPython/core/interactiveshell.py:3748: LangChainDeprecationWarning: RunnableWithMessageHistory is deprecated. Use LangGraph's built-in persistence instead.
  exec(code_obj, self.user_global_ns, self.user_ns)


In [13]:
redis_conversation.invoke(
    {'input': "나는 오라클이야."},
    config={
        'configurable' : {
            "session_id" : 'sesac_01'
        }
    }
)


AIMessage(content='안녕하세요, 오라클님! 정말 신비롭고 멋진 이름(혹은 칭호)을 가지고 계시네요. 😊\n\n미래를 내다보시는 예언자이신가요, 아니면 다른 특별한 의미가 있으신가요? 어떤 분인지 더 알려주시면 제가 더 친절하게 대화 나눌 수 있을 것 같아요! 무엇을 도와드릴까요? ✨', additional_kwargs={'reasoning_content': '*   User says: "나는 오라클이야." (I am Oracle.)\n    *   Context: The user is identifying themselves as "Oracle." This could be a roleplay, a reference to the database software, or just a playful statement.\n\n    *   Persona: Kind AI (친절한 AI).\n    *   Goal: Respond politely and engagingly.\n\n    *   *Option 1 (Literal/Software):* Treat them as the Oracle Database company or system. (Too technical, might be boring).\n    *   *Option 2 (Mythological/Fantasy):* Treat them as a prophet/oracle who sees the future. (More imaginative and fun).\n    *   *Option 3 (General/Playful):* Acknowledge it with curiosity and kindness.\n\n    *   "Hello, Oracle! That\'s an impressive name/title. Are you a prophet who can see the future, or are you referring to something else?"\n    *   Korean transla

In [2]:
from langchain_community.chat_message_histories import RedisChatMessageHistory
from langchain_core.prompts import PromptTemplate
from langchain_openai import ChatOpenAI
from langchain_ollama import ChatOllama
from langchain_core.output_parsers import StrOutputParser
from dotenv import load_dotenv
load_dotenv()
import os
from langchain_core.runnables.history import RunnableWithMessageHistory
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder


/var/folders/2v/b65115v14cvgvjztf82wr9x80000gn/T/ipykernel_75999/549146905.py:1: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.chat_message_histories import RedisChatMessageHistory


In [3]:
llm = ChatOllama(
   model="gemma4:31b-mlx",
   temperature=0.2,
   reasoning=True
)


In [4]:
REDIS_URL = "redis://localhost:6379/0"
def get_redis_history(session_id : str):
    return RedisChatMessageHistory(
        session_id = session_id,
        url=REDIS_URL
    )


prompt = ChatPromptTemplate.from_messages([ ("system", "너는 친절한 AI야"),
                        MessagesPlaceholder(variable_name="history"),
                    ("human", "{input}")])
redis_conversation = RunnableWithMessageHistory(
    prompt | llm  , get_redis_history,
    input_message_key='input',
    history_messages_key='history'
)
redis_conversation.invoke(
    {'input': "내가 했던 질문은"},
    config={
        'configurable' : {
            "session_id" : 'sesac_01'
        }
    }
)


/Users/hyeon/3_AI/.venv/lib/python3.12/site-packages/IPython/core/interactiveshell.py:3748: LangChainDeprecationWarning: RunnableWithMessageHistory is deprecated. Use LangGraph's built-in persistence instead.
  exec(code_obj, self.user_global_ns, self.user_ns)


AIMessage(content='앗, 확인해 보니 이번 대화에서는 아직 구체적인 질문을 하지 않으셨어요! 😊\n\n방금 전에는 **"나는 오라클이야"**라고 자기소개를 해주신 것뿐이라, 제가 답변해 드릴 질문이 아직 기록되어 있지 않습니다.\n\n혹시 다른 곳에서 물어보셨거나, 지금 막 생각나신 질문이 있으신가요? 어떤 것이든 편하게 말씀해 주세요! 정성껏 도와드리겠습니다. ✨', additional_kwargs={'reasoning_content': '*   User says: "내가 했던 질문은" (The question I asked was...)\n    *   Context: The user previously introduced themselves as "Oracle." This is the second turn in the conversation.\n\n    *   Looking back at the chat history:\n        1.  User: "나는 오라클이야." (I am Oracle.)\n        2.  AI: "안녕하세요, 오라클님! ..." (Hello, Oracle! ...)\n        3.  User: "내가 했던 질문은" (The question I asked was...)\n\n    *   Wait, the user *hasn\'t actually asked a specific question yet* in this session. They only made a statement about their identity.\n\n    *   Since there is no previous question in this current conversation thread, I need to politely inform the user that I don\'t see any prior questions and ask them to repeat it or tell me what they are referring 

In [6]:
!streamlit run chatbot.py

2026-07-16 14:01:43.550 Uvicorn server started on :::8501

  You can now view your Streamlit app in your browser.

  Local URL: http://localhost:8501
  Network URL: http://192.168.100.55:8501

  Help agents write better Streamlit apps?
  Install the official Streamlit skills by running streamlit skills in your terminal.

  For better performance, install the Watchdog module:

  $ xcode-select --install
  $ pip install watchdog
            
/Users/hyeon/3_AI/260716/chatbot.py:3: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.chat_message_histories import  SQLChatMessageHistory
^C
  Stopping...
